# 04 — RAG Avançado

**Módulo:** EAI_07 — IA Generativa  
**Submódulo:** 03_RAG  
**Ambiente:** `eai07` (Python 3.11)

---

## O que você vai aprender

- **Cache de embeddings** — salva o índice em disco e recarrega sem reprocessar
- **Filtro por módulo** — busca apenas dentro de EAI_01, EAI_02, etc.
- **Reranking** — reordena resultados usando o LLM como juiz
- **Histórico de conversa** — mantém contexto entre perguntas
- **Query expansion** — o LLM reformula a pergunta para melhorar a busca

---

> 💡 O RAG básico já funciona bem. Estas técnicas resolvem os casos  
> onde ele ainda erra — como a regressão linear que foi para o EAI_04.

## Setup — carrega o índice

In [5]:
import sys, os, json, pickle, time
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from dotenv import load_dotenv

sys.path.append(os.path.abspath('..'))
load_dotenv('../.env')

modelo_emb = SentenceTransformer('all-MiniLM-L6-v2')
llm = OpenAI(
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    base_url='https://api.deepseek.com'
)
LLM_MODEL = os.getenv('LLM_MODEL', 'deepseek-chat')
PROJETO_BASE = os.path.abspath('../..')

print(f'LLM   : {LLM_MODEL}')
print(f'Projeto: {PROJETO_BASE}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LLM   : deepseek-chat
Projeto: C:\Users\pcwin\Documents\Especialista_em_AI


In [6]:
# Funções do notebook anterior (copiadas para este ser independente)
ENRIQUECIMENTO = {
    'regressão linear'       : 'regressão linear, ajuste de curva, reta, mínimos quadrados, ajustar linha, coeficientes a e b',
    'mínimos quadrados'      : 'mínimos quadrados, regressão linear, ajuste de reta, least squares',
    'MSE'                    : 'MSE, Mean Squared Error, erro quadrático médio',
    'CNN'                    : 'CNN, rede convolucional, convolutional neural network, redes convolucionais, conv2D',
    'LSTM'                   : 'LSTM, Long Short-Term Memory, células de memória, gates, sequências',
    'deep learning'          : 'deep learning, aprendizado profundo, redes neurais profundas, DL',
    'ANN'                    : 'ANN, rede neural artificial, perceptron, MLP',
    'GRU'                    : 'GRU, Gated Recurrent Unit, células recorrentes',
    'KNN'                    : 'KNN, K-Nearest Neighbors, vizinhos mais próximos',
    'Random Forest'          : 'Random Forest, floresta aleatória, ensemble, árvores de decisão',
    'SVM'                    : 'SVM, Support Vector Machine',
    'embeddings'             : 'embeddings, word embeddings, vetores de palavras, representação vetorial',
    'TF-IDF'                 : 'TF-IDF, term frequency, bag of words, BoW',
    'transformers'           : 'transformers, BERT, GPT, attention, mecanismo de atenção',
    'autovalores'            : 'autovalores, autovetores, eigenvalues, PCA',
    'transformações lineares': 'transformações lineares, matrizes, rotação, escala, cisalhamento',
    'OpenCV'                 : 'OpenCV, visão computacional, processamento de imagens',
    'YOLO'                   : 'YOLO, YOLOv5, detecção de objetos, object detection',
    'reconhecimento facial'  : 'reconhecimento facial, face recognition, detecção de rostos',
    'RAG'                    : 'RAG, Retrieval Augmented Generation, recuperação de documentos, busca semântica',
    'function calling'       : 'function calling, tool calling, ferramentas, tools, agentes, modelo decide',
    'prompt engineering'     : 'prompt engineering, zero-shot, few-shot, chain-of-thought, CoT',
    'MLflow'                 : 'MLflow, rastreamento de experimentos, experiment tracking',
    'drift'                  : 'drift, data drift, monitoramento, degradação do modelo',
}

def enriquecer_chunk(texto, modulo='', arquivo=''):
    prefixo = f'[{modulo}' + (f' / {arquivo}' if arquivo else '') + '] ' if modulo else ''
    sinonimos = [v for k, v in ENRIQUECIMENTO.items() if k.lower() in texto.lower()]
    resultado = prefixo + texto
    if sinonimos:
        resultado += ' | ' + '; '.join(sinonimos)
    return resultado

def chunk_por_secao(texto):
    chunks, titulo, linhas = [], 'Introdução', []
    for linha in texto.split('\n'):
        if linha.startswith('#'):
            if linhas:
                c = ' '.join(linhas).strip()
                if c: chunks.append({'titulo': titulo, 'conteudo': c})
            titulo, linhas = linha.lstrip('#').strip(), []
        elif linha.strip():
            linhas.append(linha.strip())
    if linhas:
        c = ' '.join(linhas).strip()
        if c: chunks.append({'titulo': titulo, 'conteudo': c})
    return chunks

def processar_agent_context(conteudo, modulo):
    chunks = []
    for s in chunk_por_secao(conteudo):
        resumo = ' '.join(s['conteudo'].split()[:40])
        chunks.append({
            'chunk_busca'   : enriquecer_chunk(f"{s['titulo']}: {resumo}", modulo=modulo),
            'chunk_contexto': f"[{modulo} — {s['titulo']}]\n{s['conteudo']}",
            'titulo'        : s['titulo'],
            'modulo'        : modulo,
        })
    return chunks

def encontrar_agent_contexts(pasta_raiz):
    encontrados, ignorar = [], {'.git', 'venv', '.venv', '__pycache__', 'node_modules'}
    for raiz, dirs, arquivos in os.walk(pasta_raiz):
        dirs[:] = [d for d in dirs if d not in ignorar and not d.startswith('.')]
        if 'AGENT_CONTEXT.md' in arquivos:
            partes = raiz.replace('\\', '/').split('/')
            modulo = next((p for p in partes if p.startswith('EAI_')), os.path.basename(raiz))
            encontrados.append((modulo, os.path.join(raiz, 'AGENT_CONTEXT.md')))
    return sorted(encontrados)

print('Funções carregadas.')

Funções carregadas.


---
## 1. Cache de embeddings em disco

Reindexar 1553 chunks leva ~30 segundos. Com cache, recarrega em <1 segundo.

In [7]:
CACHE_PATH = '../data/cache/indice_rag.pkl'
os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)


def construir_indice(todos_chunks: list) -> dict:
    """Gera embeddings e monta o índice FAISS."""
    textos_busca = [c['chunk_busca'] for c in todos_chunks]
    print(f'Gerando embeddings para {len(textos_busca)} chunks...')
    t0 = time.time()
    embs = modelo_emb.encode(
        textos_busca, normalize_embeddings=True,
        show_progress_bar=True, batch_size=64
    ).astype(np.float32)
    indice_faiss = faiss.IndexFlatIP(embs.shape[1])
    indice_faiss.add(embs)
    print(f'Indexado em {time.time()-t0:.1f}s')
    return {'faiss': indice_faiss, 'chunks': todos_chunks, 'embs': embs}


def salvar_cache(indice_dict: dict, caminho: str):
    """Serializa o índice para disco."""
    # FAISS precisa de serialização própria
    faiss_bytes = faiss.serialize_index(indice_dict['faiss'])
    dados = {
        'faiss_bytes': faiss_bytes,
        'chunks'     : indice_dict['chunks'],
        'embs'       : indice_dict['embs'],
    }
    with open(caminho, 'wb') as f:
        pickle.dump(dados, f)
    tamanho = os.path.getsize(caminho) / 1024 / 1024
    print(f'Cache salvo: {caminho} ({tamanho:.1f} MB)')


def carregar_cache(caminho: str) -> dict:
    """Carrega o índice do disco."""
    with open(caminho, 'rb') as f:
        dados = pickle.load(f)
    indice_faiss = faiss.deserialize_index(dados['faiss_bytes'])
    return {'faiss': indice_faiss, 'chunks': dados['chunks'], 'embs': dados['embs']}


# Carrega do cache se existir, senão reconstrói
if os.path.exists(CACHE_PATH):
    print(f'Cache encontrado! Carregando...')
    t0 = time.time()
    INDICE = carregar_cache(CACHE_PATH)
    print(f'Carregado em {time.time()-t0:.2f}s ({len(INDICE["chunks"])} chunks)')
else:
    print('Cache não encontrado. Construindo índice...')
    todos_chunks = []
    for modulo, caminho in encontrar_agent_contexts(PROJETO_BASE):
        with open(caminho, 'r', encoding='utf-8') as f:
            conteudo = f.read()
        todos_chunks.extend(processar_agent_context(conteudo, modulo))
    INDICE = construir_indice(todos_chunks)
    salvar_cache(INDICE, CACHE_PATH)

print(f'\nÍndice pronto: {INDICE["faiss"].ntotal} chunks')

Cache não encontrado. Construindo índice...
Gerando embeddings para 1763 chunks...


Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Indexado em 60.0s
Cache salvo: ../data/cache/indice_rag.pkl (6.1 MB)

Índice pronto: 1763 chunks


---
## 2. Busca com filtro por módulo

Quando o usuário quer informações de um módulo específico, filtramos antes de buscar.

In [8]:
def buscar(query: str, top_k: int = 5, score_minimo: float = 0.3,
           filtro_modulo: str = None) -> list:
    """
    Busca semântica com filtro opcional por módulo.

    filtro_modulo: prefixo do módulo, ex: 'EAI_01', 'EAI_03'
    """
    emb_q = modelo_emb.encode([query], normalize_embeddings=True).astype(np.float32)

    if filtro_modulo:
        # Filtra chunks do módulo e recria um índice temporário
        indices_modulo = [
            i for i, c in enumerate(INDICE['chunks'])
            if c['modulo'].startswith(filtro_modulo)
        ]
        if not indices_modulo:
            return []
        embs_filtrados = INDICE['embs'][indices_modulo]
        indice_temp = faiss.IndexFlatIP(embs_filtrados.shape[1])
        indice_temp.add(embs_filtrados)
        scores, pos = indice_temp.search(emb_q, min(top_k, len(indices_modulo)))
        indices_reais = [indices_modulo[p] for p in pos[0]]
    else:
        scores_raw, pos = INDICE['faiss'].search(emb_q, top_k)
        scores = scores_raw
        indices_reais = pos[0]

    return [
        {
            'contexto': INDICE['chunks'][i]['chunk_contexto'],
            'score'   : float(scores[0][j]),
            'meta'    : {'modulo': INDICE['chunks'][i]['modulo'],
                         'titulo': INDICE['chunks'][i]['titulo']}
        }
        for j, i in enumerate(indices_reais)
        if scores[0][j] >= score_minimo
    ]


# Teste: mesma pergunta com e sem filtro
query = 'como foi implementada a regressão linear?'

print('── Sem filtro ───────────────────────────────────────────')
for r in buscar(query, top_k=3):
    print(f"  [{r['score']:.3f}] {r['meta']}")

print()
print('── Com filtro EAI_01 ────────────────────────────────────')
for r in buscar(query, top_k=3, filtro_modulo='EAI_01'):
    print(f"  [{r['score']:.3f}] {r['meta']}")

── Sem filtro ───────────────────────────────────────────
  [0.528] {'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': '5. regressao_manual.ipynb'}
  [0.523] {'modulo': 'EAI_07_AI_Generative', 'titulo': 'Query Expansion'}
  [0.520] {'modulo': 'EAI_02_Machine_Learning', 'titulo': 'Classificação'}

── Com filtro EAI_01 ────────────────────────────────────
  [0.528] {'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': '5. regressao_manual.ipynb'}
  [0.495] {'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': 'MÉTRICAS E RESULTADOS'}
  [0.493] {'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': 'Previsão'}


---
## 3. Query Expansion — o LLM melhora a busca

Antes de buscar, o LLM reformula a pergunta em termos técnicos mais precisos.  
Resolve casos como `"ajustar uma linha"` que não encontrava `"regressão linear"`.

In [9]:
def expandir_query(pergunta: str) -> str:
    """
    Usa o LLM para reformular a pergunta em termos técnicos mais precisos.
    Melhora o recall da busca semântica.
    """
    prompt = f"""\
Você é um especialista em IA. Reformule a pergunta abaixo em termos técnicos
mais precisos para melhorar uma busca semântica em documentação técnica de IA.
Inclua sinônimos e termos relacionados. Responda APENAS com a query reformulada,
sem explicações. Máximo de 2 linhas.

Pergunta original: {pergunta}
Query reformulada:"""

    response = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
        max_tokens=100
    )
    return response.choices[0].message.content.strip()


# Testa com os casos que falharam
perguntas_problematicas = [
    'como ajustar uma linha aos pontos de dados?',
    'como o modelo decide qual ferramenta chamar?',
    'como funciona o mecanismo de atenção?',
]

print('Query Expansion:\n')
for p in perguntas_problematicas:
    expandida = expandir_query(p)
    print(f'  Original  : {p}')
    print(f'  Expandida : {expandida}')
    print()

Query Expansion:

  Original  : como ajustar uma linha aos pontos de dados?
  Expandida : "técnicas de regressão linear para ajuste de curva a dados observacionais" "métodos de otimização de parâmetros para minimização de erro quadrático médio em modelos de regressão" "algoritmos de ajuste polinomial ou spline a conjuntos de pontos"

  Original  : como o modelo decide qual ferramenta chamar?
  Expandida : Mecanismo de seleção de ferramentas em modelos de linguagem: decisão de chamada de função (function calling), roteamento de ferramentas (tool routing), seleção de API, ativação de ação baseada em intenção (intent-based action triggering) e critérios de escolha entre múltiplas ferramentas.

  Original  : como funciona o mecanismo de atenção?
  Expandida : Mecanismo de atenção (attention mechanism) em transformers: funcionamento, cálculo de pesos de atenção (attention weights), self-attention, cross-attention, query-key-value (QKV), softmax, mapas de atenção (attention maps) e propagaçã

In [10]:
# Compara busca com query original vs expandida
query_original = 'como ajustar uma linha aos pontos de dados?'
query_expandida = expandir_query(query_original)

print(f'Original : {query_original}')
print(f'Expandida: {query_expandida}\n')

print('── Busca com query ORIGINAL ─────────────────────────────')
for r in buscar(query_original, top_k=2):
    print(f"  [{r['score']:.4f}] {r['meta']}")

print()
print('── Busca com query EXPANDIDA ────────────────────────────')
for r in buscar(query_expandida, top_k=2):
    print(f"  [{r['score']:.4f}] {r['meta']}")

Original : como ajustar uma linha aos pontos de dados?
Expandida: "técnicas de regressão linear para ajuste de curva a dados observados" "métodos de mínimos quadrados para aproximação de pontos" "modelagem de regressão polinomial ou linear para fitting de dataset"

── Busca com query ORIGINAL ─────────────────────────────
  [0.5619] {'modulo': 'EAI_07_AI_Generative', 'titulo': 'FAQ'}
  [0.5135] {'modulo': 'EAI_04_NLP_Classico', 'titulo': 'Conceito'}

── Busca com query EXPANDIDA ────────────────────────────
  [0.6701] {'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': '5. regressao_manual.ipynb'}
  [0.6447] {'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': 'Previsão'}


---
## 4. Reranking — o LLM reordena os resultados

Depois da busca vetorial, o LLM avalia quais chunks são realmente relevantes.

In [12]:
def rerankar(query: str, resultados: list) -> list:
    """
    Usa o LLM para reordenar os chunks recuperados por relevância real.
    Retorna os mesmos resultados mas reordenados.
    """
    if len(resultados) <= 1:
        return resultados

    # Monta prompt com os chunks numerados
    chunks_texto = '\n\n'.join(
        f"[{i+1}] {r['meta']}\n{r['contexto'][:300]}..."
        for i, r in enumerate(resultados)
    )

    prompt = f"""\
             Pergunta: {query}

             Avalie os chunks abaixo por relevância para responder a pergunta.
             Responda APENAS com os números em ordem de relevância (ex: 3,1,4,2).
             Não inclua explicações.

             Chunks:
             {chunks_texto}

             Ordem por relevância:"""

    response = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
        max_tokens=30
    )
    ordem_str = response.choices[0].message.content.strip()

    try:
        # Parseia a ordem retornada pelo LLM
        import re
        numeros = [int(x) for x in re.findall(r'\d+', ordem_str)]
        numeros = [n for n in numeros if 1 <= n <= len(resultados)]
        # Adiciona índices não mencionados ao final
        todos = list(range(1, len(resultados) + 1))
        faltando = [n for n in todos if n not in numeros]
        ordem_final = numeros + faltando
        return [resultados[i-1] for i in ordem_final]
    except Exception:
        return resultados  # fallback: retorna ordem original


# Demonstra reranking
query = 'como foi implementada a regressão linear no projeto?'
resultados = buscar(query, top_k=4)

print('── Antes do reranking ───────────────────────────────────')
for i, r in enumerate(resultados):
    print(f"  [{i+1}] score={r['score']:.3f} {r['meta']}")

rerankeados = rerankar(query, resultados)

print()
print('── Depois do reranking ──────────────────────────────────')
for i, r in enumerate(rerankeados):
    print(f"  [{i+1}] score={r['score']:.3f} {r['meta']}")

── Antes do reranking ───────────────────────────────────
  [1] score=0.581 {'modulo': 'EAI_04_NLP_Classico', 'titulo': 'Progressão de Complexidade'}
  [2] score=0.534 {'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': 'RESUMO EXECUTIVO'}
  [3] score=0.533 {'modulo': 'EAI_02_Machine_Learning', 'titulo': 'Classificação'}
  [4] score=0.518 {'modulo': 'EAI_04_NLP_Classico', 'titulo': 'Aplicação'}

── Depois do reranking ──────────────────────────────────
  [1] score=0.534 {'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': 'RESUMO EXECUTIVO'}
  [2] score=0.581 {'modulo': 'EAI_04_NLP_Classico', 'titulo': 'Progressão de Complexidade'}
  [3] score=0.533 {'modulo': 'EAI_02_Machine_Learning', 'titulo': 'Classificação'}
  [4] score=0.518 {'modulo': 'EAI_04_NLP_Classico', 'titulo': 'Aplicação'}


---
## 5. RAG com histórico de conversa

Mantém contexto entre perguntas — o assistente lembra o que foi perguntado antes.

In [13]:
SYSTEM_PROMPT = """\
    Você é o Assistente Técnico do projeto ESPECIALISTA_EM_IA de Carlos Henrique.
    Módulos: EAI_00 (Install Config), EAI_01 (Fundamentos Matemáticos), EAI_02 (Machine Learning),
    EAI_03 (Deep Learning), EAI_04 (NLP Clássico), EAI_05 (NLP Transformers),
    EAI_06 (Visão Computacional), EAI_07 (IA Generativa), EAI_08 (MLOps).
    Responda em português. Use o contexto fornecido. Seja direto e técnico.
    """
class AssistenteRAG:
    """
    Assistente Técnico com RAG avançado:
    - Query expansion automática
    - Reranking dos resultados
    - Histórico de conversa
    - Filtro opcional por módulo
    """
    def __init__(self, max_historico: int = 6):
        self.historico    = []  # lista de {role, content}
        self.max_historico = max_historico

    def responder(
        self,
        pergunta: str,
        top_k: int = 5,
        filtro_modulo: str = None,
        usar_expansion: bool = True,
        usar_reranking: bool = True,
        verbose: bool = False
    ) -> str:

        # 1. Query expansion
        query_busca = expandir_query(pergunta) if usar_expansion else pergunta
        if verbose:
            print(f'Query busca: {query_busca[:80]}...' if len(query_busca) > 80 else f'Query busca: {query_busca}')

        # 2. Busca com filtro opcional
        resultados = buscar(query_busca, top_k=top_k, filtro_modulo=filtro_modulo)

        # 3. Reranking
        if usar_reranking and len(resultados) > 1:
            resultados = rerankar(pergunta, resultados)

        if verbose:
            print(f'Chunks: {len(resultados)}')
            for r in resultados:
                print(f"  [{r['score']:.3f}] {r['meta']}")

        # 4. Monta contexto
        if resultados:
            contexto = '\n\n---\n\n'.join(r['contexto'] for r in resultados[:3])
            conteudo_usuario = f"CONTEXTO DO PROJETO:\n{contexto}\n\nPERGUNTA: {pergunta}"
        else:
            conteudo_usuario = f"Sem contexto relevante encontrado.\n\nPERGUNTA: {pergunta}"

        # 5. Chama LLM com histórico
        mensagens = [{'role': 'system', 'content': SYSTEM_PROMPT}]
        mensagens.extend(self.historico[-self.max_historico:])
        mensagens.append({'role': 'user', 'content': conteudo_usuario})

        response = llm.chat.completions.create(
            model=LLM_MODEL,
            messages=mensagens,
            temperature=0.2,
            max_tokens=600
        )
        resposta = response.choices[0].message.content

        # 6. Atualiza histórico
        self.historico.append({'role': 'user',      'content': pergunta})
        self.historico.append({'role': 'assistant',  'content': resposta})

        return resposta

    def limpar_historico(self):
        self.historico = []
        print('Histórico limpo.')


print('AssistenteRAG definido.')

AssistenteRAG definido.


In [14]:
# Testa o histórico — segunda pergunta referencia a primeira
assistente = AssistenteRAG()

perguntas_em_sequencia = [
    'Qual projeto de deep learning classificou obras de arte?',
    'Qual foi a acurácia desse projeto?',   # referencia o anterior
    'Quais técnicas de aumento de dados foram usadas nele?',  # ainda referencia
]

for pergunta in perguntas_em_sequencia:
    print(f'\n👤 {pergunta}')
    print('─' * 55)
    resposta = assistente.responder(pergunta, verbose=False)
    print(f'🤖 {resposta}')


👤 Qual projeto de deep learning classificou obras de arte?
───────────────────────────────────────────────────────
🤖 O projeto que classificou obras de arte foi o **EAI_03_Deep_Learning** com o objetivo de **classificar pinturas em estilos artísticos**. 

Especificamente, foi utilizado **Transfer Learning** com a arquitetura **MobileNetV2** (pré-treinada no ImageNet) e fine-tuning, aplicado ao dataset **WikiArt - Painter by Numbers** (6-7 estilos). O resultado alcançou aproximadamente **65% de accuracy**, com deploy via **Flask web app** para upload de imagens.

👤 Qual foi a acurácia desse projeto?
───────────────────────────────────────────────────────
🤖 Com base no contexto fornecido, a acurácia do projeto **EAI_04_NLP_Classico** foi de:

- **Modelo 1 (Sentimento)**: **95%** (0.95)
- **Modelo 2 (Sugestão)**: **98%** (0.98)

Caso a pergunta se refira ao projeto **EAI_02_Machine_Learning** (mencionado no final do contexto), o texto indica que ele apresentou **baixa acurácia preditiva*

In [15]:
# Testa filtro por módulo + expansão + reranking
assistente2 = AssistenteRAG()

print('Pergunta específica com filtro EAI_01:\n')
resposta = assistente2.responder(
    'Como calcular a reta que melhor se ajusta a dados de altura e peso?',
    filtro_modulo='EAI_01',
    verbose=True
)
print(f'\n🤖 {resposta}')

Pergunta específica com filtro EAI_01:

Query busca: "Qual algoritmo de regressão linear (mínimos quadrados ordinários, gradiente des...
Chunks: 5
  [0.542] {'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': 'Regressão Linear'}
  [0.612] {'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': '5. regressao_manual.ipynb'}
  [0.466] {'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': 'Regressão linear manual'}
  [0.595] {'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': 'Previsão'}
  [0.529] {'modulo': 'EAI_01_Fundamentos_Matemática_para_IA', 'titulo': 'MÉTRICAS E RESULTADOS'}

🤖 Para calcular a reta que melhor se ajusta aos dados de altura (x) e peso (y), siga os passos abaixo utilizando o método dos mínimos quadrados:

**1. Calcule o coeficiente angular (a):**
```python
a = (n * sum(x*y) - sum(x) * sum(y)) / (n * sum(x**2) - sum(x)**2)
```

**2. Calcule o intercepto (b):**
```python
b = (sum(y) - a * sum(x)) / n
```

**3. Equação da reta ajustada:*

---
## Resumo

| Técnica | Problema que resolve |
|---|---|
| **Cache em disco** | Reindexação lenta a cada sessão |
| **Filtro por módulo** | Busca retorna chunks de módulos errados |
| **Query Expansion** | Query em linguagem natural ≠ termos técnicos do documento |
| **Reranking** | Score vetorial não reflete relevância semântica real |
| **Histórico** | Assistente esquece o contexto da conversa |

### AssistenteRAG — o que entra no projeto final

```python
# Uso na interface Flask (próximo módulo)
assistente = AssistenteRAG(max_historico=6)

resposta = assistente.responder(
    pergunta       = request.json['pergunta'],
    filtro_modulo  = request.json.get('modulo'),  # opcional
    usar_expansion = True,
    usar_reranking = True
)
```

---